In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    chi2, f_classif, mutual_info_classif, VarianceThreshold
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
from kneed import KneeLocator
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import datetime
import gc

warnings.filterwarnings("ignore")


In [2]:
NUM_CORRIDA = 5
fecha_actual = datetime.datetime.today().strftime('%Y_%m_%d')
SUBFOLDER = f"{NUM_CORRIDA:02d}_{fecha_actual}"

REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.datetime.today().strftime('%Y%m%d')
TARGET_DIRS = ['1_vs_resto', '5_vs_resto']
PCA_SAMPLE_SIZE = 100000
PCA_RANDOM_STATE = 42

OUTPUT_PATH = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion_v2' / SUBFOLDER
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

INPUT_PATH = REPO_ROOT / 'data' / 'intermediate' / '04_featuring_v2' / RUN_ID

if not INPUT_PATH.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '04_featuring_v2').glob('*/1_vs_resto'))
    if candidates:
        INPUT_PATH = candidates[-1].parent
        print(f'Usando ultimo featuring encontrado: {INPUT_PATH}')
    else:
        raise FileNotFoundError('No se encontro featuring en data/intermediate/04_featuring_v2')

print('Rutas configuradas:')
print(f'INPUT_PATH: {INPUT_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')
print(f'TARGET_DIRS: {TARGET_DIRS}')


Usando ultimo featuring encontrado: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\04_featuring_v2\20260330
Rutas configuradas:
INPUT_PATH: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\04_featuring_v2\20260330
OUTPUT_PATH: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\05_seleccion_v2\05_2026_03_31
TARGET_DIRS: ['1_vs_resto', '5_vs_resto']


In [3]:
def seleccionar_variables(X_train, y_train, metodo, metric_path, graph_path, auto_umbral=True, nombre_dataset=""):
    if metodo == "CHI2":
        X_train_nonneg = MinMaxScaler().fit_transform(X_train)
        scores = chi2(X_train_nonneg, y_train)[0]
    elif metodo == "ANOVA":
        scores = f_classif(X_train, y_train)[0]
    elif metodo == "MI":
        scores = mutual_info_classif(X_train, y_train)
    elif metodo == "VAR":
        threshold_var = 0.01
        selector = VarianceThreshold(threshold=threshold_var)
        selector.fit(X_train)
        mask = selector.get_support()
        cols = X_train.columns[mask]
        scores_series = pd.Series(selector.variances_, index=X_train.columns)

        with PdfPages(os.path.join(graph_path, f"varianza_{nombre_dataset}_{metodo}.pdf")) as pdf:
            plt.figure(figsize=(10,4))
            scores_series.sort_values(ascending=False).plot(kind='bar')
            plt.title(f"Varianza de variables - {nombre_dataset} [{metodo}]")
            plt.ylabel("Varianza")
            plt.tight_layout()
            pdf.savefig(); plt.close()
        return cols, scores_series
    else:
        raise ValueError("Método de selección no reconocido")

    scores_series = pd.Series(scores, index=X_train.columns)
    scores_sorted = scores_series.sort_values(ascending=False)

    with PdfPages(os.path.join(graph_path, f"codo_{nombre_dataset}_{metodo}.pdf")) as pdf:
        plt.figure(figsize=(10,4))
        plt.plot(range(len(scores_sorted)), scores_sorted.values, marker='o')
        plt.xlabel("Variables ordenadas")
        plt.ylabel("Score")
        plt.title(f"Curva de importancia de variables - {nombre_dataset} [{metodo}]")
        plt.tight_layout()
        pdf.savefig(); plt.close()

    if auto_umbral:
        knee = KneeLocator(
            range(len(scores_sorted)), scores_sorted.values,
            curve='convex', direction='decreasing'
        )
        if knee.knee is not None and knee.knee > 0:
            best_n = knee.knee
            selected_features = scores_sorted.index[:best_n]
            codo_info = {
                "metodo": metodo,
                "dataset": nombre_dataset,
                "codo_posicion": int(best_n),
                "score_codo": float(scores_sorted.values[best_n-1]),
                "total_variables": len(scores_sorted)
            }
        else:
            threshold = 0.05 * np.nanmax(scores)
            selected_mask = scores >= threshold
            selected_features = X_train.columns[selected_mask]
            codo_info = {
                "metodo": metodo,
                "dataset": nombre_dataset,
                "codo_posicion": None,
                "score_codo": None,
                "umbral_fallback": float(threshold),
                "total_variables": len(scores_sorted)
            }
    else:
        threshold = 0.05 * np.nanmax(scores)
        selected_mask = scores >= threshold
        selected_features = X_train.columns[selected_mask]
        codo_info = {
            "metodo": metodo,
            "dataset": nombre_dataset,
            "codo_posicion": None,
            "score_codo": None,
            "umbral_fallback": float(threshold),
            "total_variables": len(scores_sorted)
        }

    pd.DataFrame([codo_info]).to_csv(
        os.path.join(metric_path, f"codo_{nombre_dataset}_{metodo}.csv"), index=False
    )

    return selected_features, scores_series


In [4]:
def aplicar_pca(X_train, X_test, metric_path, graph_path, n_components, modo="fijo", nombre_dataset=""):
    sample_size = min(PCA_SAMPLE_SIZE, len(X_train))
    if sample_size < len(X_train):
        X_train_sample = X_train.sample(n=sample_size, random_state=PCA_RANDOM_STATE)
    else:
        X_train_sample = X_train

    scaler = StandardScaler()
    X_train_sample_scaled = scaler.fit_transform(X_train_sample)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if modo == "fijo":
        pca_full = PCA(svd_solver="randomized", random_state=PCA_RANDOM_STATE).fit(X_train_sample_scaled)
        pca = PCA(n_components=n_components, svd_solver="randomized", random_state=PCA_RANDOM_STATE)
        pca.fit(X_train_sample_scaled)
        X_train_pca = pca.transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_components)])
        n_opt = n_components
    elif modo == "optimo":
        pca_full = PCA(random_state=PCA_RANDOM_STATE).fit(X_train_sample_scaled)
        cumsum = np.cumsum(pca_full.explained_variance_ratio_)
        n_opt = np.argmax(cumsum >= 0.95) + 1
        pca = PCA(n_components=n_opt, svd_solver="randomized", random_state=PCA_RANDOM_STATE)
        pca.fit(X_train_sample_scaled)
        X_train_pca = pca.transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_opt)])
    else:
        raise ValueError("Modo PCA no reconocido")

    with PdfPages(os.path.join(graph_path, f"pca_{nombre_dataset}_{modo}.pdf")) as pdf:
        plt.figure(figsize=(10,4))
        plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
        plt.axhline(0.95, color='r', linestyle='--', label='95% varianza')
        plt.xlabel("N° componentes")
        plt.ylabel("Varianza explicada acumulada")
        plt.title(f"PCA - Varianza explicada acumulada [{nombre_dataset}] (muestra={sample_size})")
        plt.legend()
        plt.tight_layout()
        pdf.savefig(); plt.close()

    pca_info = {
        "dataset": nombre_dataset,
        "modo": modo,
        "componentes_optimos": n_opt,
        "varianza_95": float(np.cumsum(pca_full.explained_variance_ratio_)[n_opt-1]),
        "sample_size": int(sample_size),
        "sampled": bool(sample_size < len(X_train))
    }
    pd.DataFrame([pca_info]).to_csv(
        os.path.join(metric_path, f"pca_{nombre_dataset}_{modo}.csv"), index=False
    )

    return pd.DataFrame(X_train_pca), pd.DataFrame(X_test_pca), scores


In [5]:
# Procesar todas las combinaciones
metodos = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]
resumen_comparativo_global = []

for target_name in TARGET_DIRS:
    target_input_path = INPUT_PATH / target_name
    if not target_input_path.exists():
        print(f"No se encontro rama {target_name}; se omite.")
        continue

    target_output_path = OUTPUT_PATH / target_name
    metric_path = target_output_path / 'metricas'
    graph_path = target_output_path / 'graficos'
    target_output_path.mkdir(parents=True, exist_ok=True)
    metric_path.mkdir(parents=True, exist_ok=True)
    graph_path.mkdir(parents=True, exist_ok=True)

    print(f"\nProcesando target: {target_name}")

    resumen_comparativo = []
    procesados = set()

    for carpeta in os.listdir(target_input_path):
        carpeta_path = target_input_path / carpeta
        if not carpeta_path.is_dir():
            continue

        for tipo in ["ORIGINAL", "FE"]:
            nombre_dataset = f"{carpeta}_{tipo}"
            if nombre_dataset in procesados:
                continue
            procesados.add(nombre_dataset)
            try:
                X_train = pd.read_parquet(carpeta_path / f"X_train_{carpeta}_{tipo}.parquet")
                X_test = pd.read_parquet(carpeta_path / f"X_test_{carpeta}_{tipo}.parquet")
                y_train = pd.read_parquet(carpeta_path / f"y_train_{carpeta}_{tipo}.parquet")
                y_test = pd.read_parquet(carpeta_path / f"y_test_{carpeta}_{tipo}.parquet")
                X_train = X_train.astype(np.float32)
                X_test = X_test.astype(np.float32)
            except Exception as e:
                print(f"Error cargando {target_name}/{nombre_dataset}: {e}")
                continue

            print(f"Procesando {target_name} | {nombre_dataset}...")

            for metodo in metodos:
                if metodo == "CHI2" and (X_train < 0).any().any():
                    print(f"Saltando CHI2 por contener negativos en {target_name}/{nombre_dataset}")
                    continue

                if metodo == "ALL":
                    selected_columns = X_train.columns
                    scores = pd.Series([1.0] * len(selected_columns), index=selected_columns)
                elif metodo == "VAR":
                    selected_columns, scores = seleccionar_variables(
                        X_train, y_train.values.ravel(), metodo, metric_path, graph_path,
                        auto_umbral=False, nombre_dataset=nombre_dataset
                    )
                else:
                    try:
                        selected_columns, scores = seleccionar_variables(
                            X_train, y_train.values.ravel(), metodo, metric_path, graph_path,
                            auto_umbral=True, nombre_dataset=nombre_dataset
                        )
                    except Exception as e:
                        print(f"Error aplicando {metodo} en {target_name}/{nombre_dataset}: {e}")
                        continue

                scores.to_csv(metric_path / f"metricas_{nombre_dataset}_{metodo}.csv", header=["score"])

                X_train_sel = X_train[selected_columns]
                X_test_sel = X_test[selected_columns.intersection(X_test.columns)]
                X_train_sel.to_parquet(target_output_path / f"X_train_{nombre_dataset}_{metodo}.parquet")
                X_test_sel.to_parquet(target_output_path / f"X_test_{nombre_dataset}_{metodo}.parquet")
                y_train.to_parquet(target_output_path / f"y_train_{nombre_dataset}_{metodo}.parquet")
                y_test.to_parquet(target_output_path / f"y_test_{nombre_dataset}_{metodo}.parquet")

                print(f"{target_name} | {nombre_dataset} - {metodo}: {len(selected_columns)} variables seleccionadas")

                fila = {
                    "target": target_name,
                    "dataset": nombre_dataset,
                    "metodo": metodo,
                    "columnas_seleccionadas": len(selected_columns)
                }
                resumen_comparativo.append(fila)
                resumen_comparativo_global.append(fila)
                del X_train_sel, X_test_sel, scores
                gc.collect()

            for pca_tipo, modo in [("PCA30", "fijo"), ("PCAopt", "optimo")]:
                try:
                    X_train_pca, X_test_pca, pca_scores = aplicar_pca(
                        X_train, X_test, metric_path, graph_path,
                        n_components=30, modo=modo, nombre_dataset=nombre_dataset
                    )

                    X_train_pca.to_parquet(target_output_path / f"X_train_{nombre_dataset}_{pca_tipo}.parquet")
                    X_test_pca.to_parquet(target_output_path / f"X_test_{nombre_dataset}_{pca_tipo}.parquet")
                    y_train.to_parquet(target_output_path / f"y_train_{nombre_dataset}_{pca_tipo}.parquet")
                    y_test.to_parquet(target_output_path / f"y_test_{nombre_dataset}_{pca_tipo}.parquet")

                    pca_scores.to_csv(
                        metric_path / f"metricas_{nombre_dataset}_{pca_tipo}.csv",
                        header=["explained_variance_ratio"]
                    )

                    print(f"{target_name} | {nombre_dataset} - {pca_tipo}: {X_train_pca.shape[1]} componentes")
                    fila = {
                        "target": target_name,
                        "dataset": nombre_dataset,
                        "metodo": pca_tipo,
                        "columnas_seleccionadas": X_train_pca.shape[1]
                    }
                    resumen_comparativo.append(fila)
                    resumen_comparativo_global.append(fila)
                    del X_train_pca, X_test_pca, pca_scores
                    gc.collect()
                except Exception as e:
                    print(f"Error en PCA {pca_tipo} para {target_name}/{nombre_dataset}: {e}")

            del X_train, X_test, y_train, y_test
            gc.collect()

    df_resumen_target = pd.DataFrame(resumen_comparativo)
    df_resumen_target.to_csv(target_output_path / "resumen_cantidad_variables.csv", index=False)

if resumen_comparativo_global:
    df_resumen_global = pd.DataFrame(resumen_comparativo_global)
    df_resumen_global.to_csv(OUTPUT_PATH / "resumen_cantidad_variables_global.csv", index=False)
else:
    print("No se encontraron datasets para procesar.")



Procesando target: 1_vs_resto
Procesando 1_vs_resto | MaxAbs_ORIGINAL...
Saltando CHI2 por contener negativos en 1_vs_resto/MaxAbs_ORIGINAL
1_vs_resto | MaxAbs_ORIGINAL - ANOVA: 80 variables seleccionadas
1_vs_resto | MaxAbs_ORIGINAL - MI: 42 variables seleccionadas
1_vs_resto | MaxAbs_ORIGINAL - VAR: 164 variables seleccionadas
1_vs_resto | MaxAbs_ORIGINAL - ALL: 571 variables seleccionadas
1_vs_resto | MaxAbs_ORIGINAL - PCA30: 30 componentes
1_vs_resto | MaxAbs_ORIGINAL - PCAopt: 480 componentes
Procesando 1_vs_resto | MaxAbs_FE...
Saltando CHI2 por contener negativos en 1_vs_resto/MaxAbs_FE
1_vs_resto | MaxAbs_FE - ANOVA: 230 variables seleccionadas
1_vs_resto | MaxAbs_FE - MI: 33 variables seleccionadas
1_vs_resto | MaxAbs_FE - VAR: 377 variables seleccionadas
1_vs_resto | MaxAbs_FE - ALL: 1180 variables seleccionadas
1_vs_resto | MaxAbs_FE - PCA30: 30 componentes
1_vs_resto | MaxAbs_FE - PCAopt: 485 componentes
Procesando 1_vs_resto | MinMax_ORIGINAL...
1_vs_resto | MinMax_ORIGIN

In [6]:
# Visualizacion rapida del resumen global
if "df_resumen_global" in globals():
    display(df_resumen_global)
else:
    resumen_file = OUTPUT_PATH / "resumen_cantidad_variables_global.csv"
    if resumen_file.exists():
        display(pd.read_csv(resumen_file))
    else:
        print("Primero ejecuta la celda principal de seleccion para generar el resumen.")


,target,dataset,metodo,columnas_seleccionadas
0,1_vs_resto,MaxAbs_ORIGINAL,ANOVA,80
1,1_vs_resto,MaxAbs_ORIGINAL,MI,42
2,1_vs_resto,MaxAbs_ORIGINAL,VAR,164
3,1_vs_resto,MaxAbs_ORIGINAL,ALL,571
4,1_vs_resto,MaxAbs_ORIGINAL,PCA30,30
...,...,...,...,...
113,5_vs_resto,Standard_FE,MI,24
114,5_vs_resto,Standard_FE,VAR,1186
115,5_vs_resto,Standard_FE,ALL,1191
116,5_vs_resto,Standard_FE,PCA30,30
